# Clustering of TOP 5 European Leagues Players - 2022/23
**Authors:** Cezary Kuźmowicz, Filip Żebrowski, Dariusz Doktorski

This notebook extends the 2021/22 reproduction analysis to the 2022/23 season. We proceed the same way as before: use the same feature set, preprocessing rules, clustering method, and evaluation plots, but load the newer season data.

[dataset used for analysis](https://www.kaggle.com/datasets/vivovinco/20222023-football-player-stats)

In [9]:
from IPython.display import display

from football_clustering import (
    AnalysisConfig,
    add_cluster_labels,
    add_pca_components,
    cluster_feature_means,
    configure_notebook,
    evaluate_clusters,
    filter_by_minutes,
    fit_kmeans,
    hopkins_statistic,
    load_player_stats,
    plot_cluster_evaluation,
    plot_correlation_matrix,
    plot_distributions,
    plot_pca_clusters,
    plot_silhouette_profile,
    prepare_visualization_data,
    scale_features,
    select_features,
    validate_no_missing_values,
)

## Setup
Only the input file changes here. The shared configuration defaults and reusable analysis functions are defined in `football_clustering.py`.

In [10]:
FILE_PATH = "data/2022-2023 Football Player Stats.csv"

config = AnalysisConfig(file_path=FILE_PATH)

SELECTED_FEATURES = list(config.selected_features)
VISUALIZATION_FEATURES = list(config.visualization_features)

configure_notebook(config.random_seed)

## Data Preparation
We load the 2022/23 data and apply the same minimum-minutes filter used in the first analysis.

In [11]:
# Load the dataset using shared CSV formatting parameters.
raw_stats = load_player_stats(config)

In [12]:
# Filter players based on the configured minimum minutes played.
stats_filtered = filter_by_minutes(raw_stats, config.min_minutes_played)

In [13]:
display(stats_filtered.head())
display(stats_filtered.describe())
print(stats_filtered.info())

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Off,Crs,TklW,PKwon,PKcon,OG,Recov,AerWon,AerLost,AerWon%
0,1,Brenden Aaronson,USA,MFFW,Leeds United,Premier League,22,2000,20,19,...,0.17,2.54,0.51,0.0,0.0,0.00,4.86,0.34,1.19,22.2
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,35,1987,22,22,...,0.05,0.18,1.59,0.0,0.0,0.00,6.64,2.18,1.23,64.0
2,3,Himad Abdelli,FRA,MFFW,Angers,Ligue 1,23,1999,14,8,...,0.00,1.05,1.40,0.0,0.0,0.00,8.14,0.93,1.05,47.1
3,4,Salis Abdul Samed,GHA,MF,Lens,Ligue 1,22,2000,20,20,...,0.00,0.35,0.80,0.0,0.0,0.05,6.60,0.50,0.50,50.0
4,5,Laurent Abergel,FRA,MF,Lorient,Ligue 1,30,1993,15,15,...,0.00,0.23,2.02,0.0,0.0,0.00,6.51,0.31,0.39,44.4


,Rk,Age,Born,MP,Starts,Min,90s,Goals,Shots,SoT,...,Off,Crs,TklW,PKwon,PKcon,OG,Recov,AerWon,AerLost,AerWon%
count,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,...,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000,2061.000000
mean,1337.290150,26.601164,1995.567686,14.580786,10.917031,971.866570,10.799272,1.320233,1.202077,0.397351,...,0.188326,1.594289,0.921121,0.011664,0.015803,0.004100,4.890112,1.285939,1.348559,48.143183
std,772.673299,4.240017,4.250175,5.252729,6.178840,513.420707,5.704756,2.214323,1.007596,0.427438,...,0.316761,1.913114,0.601791,0.046284,0.050258,0.025748,1.926001,1.168281,1.111843,21.472312
min,1.000000,16.000000,1981.000000,2.000000,0.000000,180.000000,2.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,679.000000,23.000000,1993.000000,11.000000,5.000000,521.000000,5.800000,0.000000,0.410000,0.050000,...,0.000000,0.170000,0.490000,0.000000,0.000000,0.000000,3.720000,0.480000,0.670000,33.300000
50%,1329.000000,26.000000,1996.000000,16.000000,11.000000,954.000000,10.600000,0.000000,0.930000,0.270000,...,0.050000,0.860000,0.880000,0.000000,0.000000,0.000000,5.050000,1.000000,1.110000,50.000000
75%,2000.000000,30.000000,1999.000000,19.000000,16.000000,1393.000000,15.500000,2.000000,1.820000,0.630000,...,0.250000,2.420000,1.290000,0.000000,0.000000,0.000000,6.170000,1.780000,1.710000,60.600000
max,2689.000000,41.000000,2006.000000,23.000000,23.000000,2070.000000,23.000000,25.000000,5.550000,2.500000,...,2.800000,11.300000,4.000000,0.870000,0.480000,0.500000,11.300000,14.500000,9.400000,100.000000


<class 'pandas.core.frame.DataFrame'>
Index: 2061 entries, 0 to 2688
Columns: 124 entries, Rk to AerWon%
dtypes: float64(112), int64(7), object(5)
memory usage: 2.0+ MB
None
